# Camada Gold

**Bases deste notebook**
- `pesquisa_satisfacao.csv`
- `custos.csv`

## 1. Pesquisa de Satisfação
Consulta realizada para obter a visualização da distribuição das notas, da quantidade de respostas e do percentual de cada nota. Registros com notas inválidas ou nulas também são contabilizados, para não esconder problemas de qualidade.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_schema = "vcredit.silver"
gold_schema = "vcredit.gold"

df_pesquisa = spark.table(f"{silver_schema}.pesquisa_satisfacao")

df_gold_satisfacao = (
    df_pesquisa
    .groupBy("nota_atendimento")
    .agg(
        F.count("*").alias("quantidade_respostas"),
        F.sum(
            F.when(
                F.coalesce(F.col("nota_valida"), F.lit(False)),
                1
            ).otherwise(0)
        ).alias("respostas_validas")
    )
)

total_respostas = df_pesquisa.count()
df_gold_satisfacao = df_gold_satisfacao.withColumn(
    "total_respostas",
    F.lit(total_respostas)
).withColumn(
    "percentual_respostas",
    F.round(
        F.col("quantidade_respostas") /
        F.col("total_respostas") * 100,
        2
    )
)
(
    df_gold_satisfacao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.resumo_satisfacao")
)

display(spark.table(f"{gold_schema}.resumo_satisfacao").orderBy("nota_atendimento"))

## 2. Custos
Consulta dos custos voltado para encontrar: custo total válido, média, mínimo, máximo e quantidade de registros inválidos.

In [0]:
df_custos = spark.table(f"{silver_schema}.custos")

df_gold_custos = df_custos.agg(
    F.count("*").alias("total_registros"),
    F.sum(
        F.when(
            F.coalesce(F.col("custo_valido"), F.lit(False)),
            1
        ).otherwise(0)
    ).alias("registros_validos"),
    F.sum(
        F.when(
            ~F.coalesce(F.col("custo_valido"), F.lit(False)),
            1
        ).otherwise(0)
    ).alias("registros_invalidos"),
    F.sum(
        F.when(
            F.coalesce(F.col("custo_valido"), F.lit(False)),
            F.col("custo")
        )
    ).alias("custo_total"),
    F.avg(
        F.when(
            F.coalesce(F.col("custo_valido"), F.lit(False)),
            F.col("custo")
        )
    ).alias("custo_medio"),
    F.min(
        F.when(
            F.coalesce(F.col("custo_valido"), F.lit(False)),
            F.col("custo")
        )
    ).alias("menor_custo"),
    F.max(
        F.when(
            F.coalesce(F.col("custo_valido"), F.lit(False)),
            F.col("custo")
        )
    ).alias("maior_custo")
)
(
    df_gold_custos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.resumo_custos")
)

display(spark.table(f"{gold_schema}.resumo_custos"))